# บทที่ 13 - ความจำของเอเย่นต์


## การตั้งค่า

สมุดบันทึกนี้สาธิตวิธีสร้างตัวแทนจองการเดินทางด้วย **หน่วยความจำถาวร** โดยใช้ **Microsoft Agent Framework** (MAF)

คุณจะได้เรียนรู้ว่าหน่วยความจำตัวแทนประเภทต่าง ๆ — ทำงาน, ระยะสั้น, และระยะยาว — ส่งผลอย่างไรต่อการที่ตัวแทนเก็บรักษาและใช้ข้อมูลข้ามการสนทนา

**สิ่งที่ต้องมี:**
- โปรเจกต์ Microsoft Foundry ที่มีโมเดลแชทที่ถูกนำไปใช้แล้ว (เช่น `gpt-5-mini`)
- เข้าสู่ระบบด้วย Azure CLI — รัน `az login` ในเทอร์มินัลของคุณ
- `AZURE_AI_PROJECT_ENDPOINT` — จุดสิ้นสุดโปรเจกต์ Microsoft Foundry ของคุณ
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ชื่อของโมเดลที่คุณได้นำไปใช้


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ประเภทของหน่วยความจำของเอเจนต์

เอเจนต์ AI สามารถใช้ประโยชน์จากหน่วยความจำประเภทต่าง ๆ ซึ่งแต่ละประเภทมีวัตถุประสงค์ที่แตกต่างกัน:

### หน่วยความจำชั่วคราว (Working Memory)
ข้อความในหัวข้อสนทนาเอง — ข้อความที่แลกเปลี่ยนกันในเซสชันเดียวกัน เอเจนต์สามารถอ้างอิงกลับไปยังข้อความก่อนหน้านี้ในหัวข้อเดียวกันได้เพื่อรักษาความสอดคล้อง ใน MAF นี้จะถูกสร้างผ่าน **`agent.create_session()`** ซึ่งจะคืนค่าเป็น `AgentSession`

### หน่วยความจำระยะสั้น (Short-Term Memory)
ข้อมูลที่คงอยู่ระหว่างการทำงานหรือเซสชันแต่ไม่ถูกเก็บถาวร เช่น เอเจนต์อาจรวบรวมข้อเท็จจริงระหว่างการสนทนาวางแผนหลายรอบและใช้ข้อมูลเหล่านั้นเพื่อผลิตแผนการเดินทางสุดท้าย

### หน่วยความจำระยะยาว (Long-Term Memory)
ความชอบและข้อเท็จจริงที่คงอยู่ **ข้ามเซสชัน** ผู้ใช้ที่กลับมาไม่ควรต้องบอกข้อจำกัดด้านอาหารหรือสไตล์การเดินทางซ้ำ ๆ หน่วยความจำระยะยาวมักจะถูกสนับสนุนโดยที่เก็บภายนอก — ฐานข้อมูล ไฟล์ หรือดัชนีเวกเตอร์ — และแสดงผลต่อเอเจนต์ผ่านทางเครื่องมือ


## หน่วยความจำชั่วคราวด้วยเซสชัน

รูปแบบที่ง่ายที่สุดของหน่วยความจำคือเซสชันการสนทนา เมื่อคุณส่งเซสชันเดียวกัน (สร้างผ่าน `agent.create_session()`) ไปยังการเรียกใช้งาน `agent.run()` ติดต่อกัน ตัวแทนจะเห็นประวัติทั้งหมดของการสนทนานั้นและสามารถเรียกคืนรายละเอียดก่อนหน้าได้

มาสร้างตัวแทนท่องเที่ยวและสาธิตหน่วยความจำชั่วคราวกันเถอะ


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

เจ้าหน้าที่จำงบประมาณได้อย่างถูกต้องเนื่องจากทั้งสองข้อความแชร์เซสชันเดียวกัน นี่คือ **หน่วยความจำทำงาน** — ซึ่งมีอยู่เพียงช่วงเวลาของเซสชันเท่านั้น

### จะเกิดอะไรขึ้นกับกระทู้ใหม่?

หากเราสร้างเซสชัน **ใหม่** เจ้าหน้าที่จะไม่จำบทสนทนาก่อนหน้านี้ได้เลย:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## แบบจำลองความจำระยะยาว

เพื่อจดจำความชอบของผู้ใช้ **ข้ามเซสชัน** เราต้องมีที่เก็บข้อมูลแบบถาวรที่อยู่นอกเหนือจากกระทู้สนทนา ตัวแทนเข้าถึงที่เก็บนี้ผ่านทาง **เครื่องมือ** — ฟังก์ชันที่มันสามารถเรียกใช้เพื่อบันทึกและดึงข้อมูล

ด้านล่างนี้เราจะสร้างที่เก็บความชอบในหน่วยความจำอย่างง่าย (ในระบบจริงคุณควรสำรองข้อมูลนี้ด้วยฐานข้อมูลหรือตัวชี้วัดเวคเตอร์) และเปิดเผยเป็นเครื่องมือที่ตัวแทนสามารถใช้ได้

### สถาปัตยกรรม
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### สถานการณ์ที่ 1 — ผู้ใช้ครั้งแรกจองทริปวันครบรอบ

ซาร่าห์มาเยือนเป็นครั้งแรก ตัวแทนควรจัดเก็บความชอบของเธอผ่านเครื่องมือและใช้เพื่อแนะนำโรงแรม


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### สถานการณ์ที่ 2 — ซาร่าห์กลับมาอีกหลายสัปดาห์ต่อมา

ซาร่าห์เริ่ม **หัวข้อใหม่โดยสิ้นเชิง** (จำลองเซสชันใหม่) หน่วยความจำการทำงานว่างเปล่า แต่ตัวเก็บข้อมูลความชอบระยะยาวยังคงมีข้อมูลของเธออยู่ ตัวแทนควรดึงข้อมูลนั้นมาใช้เพื่อปรับแต่งคำแนะนำให้เหมาะกับบุคคล


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้เกี่ยวกับสามประเภทของหน่วยความจำตัวแทนและวิธีการใช้งานร่วมกับ Microsoft Agent Framework:

| ประเภทหน่วยความจำ | กลไก MAF | ระยะเวลา |
|---|---|---|
| **ทำงาน** | `agent.create_session()` | การสนทนาเดียว |
| **ระยะสั้น** | บริบทสะสมภายในเธรด | งาน / เซสชันเดียว |
| **ระยะยาว** | ที่เก็บภายนอกเข้าถึงผ่านฟังก์ชัน `@tool` | ข้ามเซสชัน |

### ข้อสรุปสำคัญ
1. **`agent.create_session()`** ให้หน่วยความจำทำงาน — ตัวแทนเห็นประวัติการสนทนาเต็มภายในเซสชัน
2. **เซสชันใหม่จะสูญเสียบริบท** — หากไม่มีหน่วยความจำระยะยาว ตัวแทนไม่สามารถจำการสนทนาในอดีตได้
3. **ฟังก์ชัน `@tool`** เป็นสะพานเชื่อม — ช่วยให้ตัวแทนบันทึกและดึงข้อมูลจากที่เก็บข้อมูลถาวร
4. **การปรับแต่งเฉพาะตัวพัฒนาขึ้นตามเวลา** — ยิ่งเก็บความชื่นชอบมากเท่าไร การแนะนำของตัวแทนก็จะยิ่งดีขึ้นเท่านั้น

### การใช้งานจริง
- **บริการลูกค้า**: จำประวัติและความชอบของลูกค้า
- **ผู้ช่วยส่วนตัว**: รักษาบริบทข้ามวันหรือสัปดาห์
- **การดูแลสุขภาพ**: ติดตามข้อมูลและความชอบของผู้ป่วย
- **อีคอมเมิร์ซ**: ช็อปปิ้งที่ปรับตามประวัติส่วนตัว

### ขั้นตอนถัดไป
- แทนที่ dict ในหน่วยความจำด้วยฐานข้อมูลหรือที่เก็บเวกเตอร์ (เช่น Azure AI Search)
- เพิ่มเวลาหมดอายุของหน่วยความจำสำหรับข้อมูลที่ต้องระวังเวลา
- สร้างระบบตัวแทนหลายตัวพร้อมหน่วยความจำที่ใช้ร่วมกัน
- สำรวจสมุดบันทึก Cognee สำหรับหน่วยความจำที่สนับสนุนด้วยกราฟความรู้


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
